# Graph Neural Networks for Feynman Diagram Amplitudes

## AI4Physics Learning Workshop — Uppsala University, April 2026

**Tutorial:** Geometric Deep Learning: Hands-on

### The Big Idea

**Feynman diagrams are literally graphs.** This notebook shows that if you represent a Feynman diagram as a graph and pass messages between its vertices, you can learn to predict scattering amplitudes. The architecture mirrors how a physicist computes amplitudes by hand:

- **Vertices of the diagram** → nodes of the graph (each carrying vertex-type information)
- **Propagators & external legs** → edges of the graph (each carrying particle-type information: mass, spin, hypercharge, colour, ...)
- **Message passing** → local update rules that propagate information across vertices, just like Feynman rules propagate factors across a diagram

The final graph-level prediction is the squared matrix element $|\mathcal{M}|^2$.

```
    e⁻ ──●────●── μ⁻                 ●─────●
         │ γ* │                      │  e  │
    e⁺ ──●────●── μ⁺         ⇒       ●─────●
       (Feynman diagram)              (graph)
```

### What We'll Do

1. **Generate a dataset from scratch** — tree-level QED 2→2 scattering. Analytic formulas for $|\mathcal{M}|^2$, graphs built from explicit diagram topology.
2. **Baseline: MLP on Mandelstam variables $(s, t, u)$** — works well on a fixed process.
3. **GNN: GAT on the diagram graph** — works well on *multiple* processes at once, because the graph structure tells it which process it's looking at.
4. **Generalization test** — show that the MLP breaks down when topology varies, while the GNN captures the structure.

### Based On

[Mitchell (2022), "Learning Feynman Diagrams using Graph Neural Networks"](https://arxiv.org/abs/2211.15348) — [matbun/Feynman-GNN](https://github.com/matbun/Feynman-GNN). Our pipeline is a pedagogically simplified rewrite of that repository's data generator and model, adapted for a 90-minute hands-on session.

## 1. Setup & Imports

We use:
- **`torch`** + **`torch_geometric`** for the GNN
- **`networkx`** + **`matplotlib`** for visualizing diagrams
- **`numpy`** / **`pandas`** for the dataset

Everything runs on CPU in a few minutes. GPU is optional.

In [ ]:
# Install dependencies (uncomment both lines if running in Colab)
# !pip install -q uv && uv pip install torch_geometric networkx

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_add_pool

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 2. Physics Primer: QED 2→2 Tree-Level Scattering

We'll work with two classic QED processes at tree level:

| Process | Channels | Matrix Element |
|---------|----------|----------------|
| $e^+ e^- \to \mu^+ \mu^-$ | $s$ only | $\|\mathcal{M}\|^2 = \frac{e^4}{2}\left[(1+\cos\theta)^2 + (1-\cos\theta)^2\right]$ |
| $e^+ e^- \to e^+ e^-$ (Bhabha) | $s + t$ | $\|\mathcal{M}\|^2 = 2 e^4 \left[\frac{u^2+t^2}{s^2} + \frac{u^2+s^2}{t^2} + \frac{2u^2}{s\,t}\right]$ |

with Mandelstam invariants $s = (p_1+p_2)^2$, $t = (p_1-p_3)^2$, $u = (p_1-p_4)^2$.

The physics of the exercise: **the first process has one diagram topology, the second has two.** Any model we train will need to handle this topological difference. That's where the graph representation will pay off.

In [ ]:
# === Physical constants (natural units, MeV) ===
# Extracted from matbun/Feynman-GNN: Dataset_generator.ipynb

# Lepton masses
m_e  = 0.5110
m_mu = 105.6583755

# Couplings
alpha_QED = 1/137
alpha_W   = 1e-6
alpha_S   = 1.0

q_e = math.sqrt(4 * math.pi * alpha_QED)   # electromagnetic coupling

print(f"m_e  = {m_e} MeV")
print(f"m_mu = {m_mu} MeV")
print(f"α_QED = {alpha_QED:.6f}")
print(f"e     = {q_e:.4f}")

In [ ]:
# === Tree-level QED matrix elements (analytic) ===
# Inputs: centre-of-mass 3-momentum p (MeV), scattering angle theta (rad)

def Mfi_ee_mumu(p, theta):
    """|M|^2 for e+ e- -> mu+ mu- (s-channel only)."""
    return ((q_e**2*(1+np.cos(theta)))**2 + (q_e**2*(1-np.cos(theta)))**2) / 2

def Mfi_bhabha(p, theta):
    """|M|^2 for e+ e- -> e+ e- (Bhabha, s+t channels)."""
    s = 4 * (m_e**2 + p**2)
    t = 2 * m_e**2 - 2 * p**2 * (1 - np.cos(theta))
    u = 2 * m_e**2 - 2 * p**2 * (1 + np.cos(theta))
    return 2 * q_e**4 * ((u**2 + t**2)/s**2 + (u**2 + s**2)/t**2 + 2*u**2/(s*t))

# Quick visualization of the two processes
theta = np.linspace(0.05, np.pi - 0.05, 200)
p0 = 1e4  # 10 GeV

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(theta, Mfi_ee_mumu(p0, theta), color='steelblue')
axes[0].set_title(r"$e^+e^-\to\mu^+\mu^-$" + f" (p = {p0/1000:.0f} GeV)")
axes[0].set_xlabel(r"$\theta$ (rad)")
axes[0].set_ylabel(r"$|\mathcal{M}|^2$")

axes[1].semilogy(theta, Mfi_bhabha(p0, theta), color='darkorange')
axes[1].set_title(r"$e^+e^-\to e^+e^-$ Bhabha")
axes[1].set_xlabel(r"$\theta$ (rad)")
axes[1].set_ylabel(r"$|\mathcal{M}|^2$ (log)")

plt.tight_layout()
plt.show()

print("Note: Bhabha diverges at small angles due to the t-channel photon pole.")

## 3. Feynman Diagrams as Graphs

Now we build the graph representation. Adapted from `matbun/Feynman-GNN/Dataset_generator.ipynb`.

### Node features (3-dim one-hot)
- `[1, 0, 0]` — initial-state vertex
- `[0, 1, 0]` — virtual (internal) vertex
- `[0, 0, 1]` — final-state vertex

### Edge features (12-dim particle descriptor)
Each edge carries the quantum numbers of the particle it represents:
`[m, S, LIW₃, LY, RIW₃, RY, c_r, c_g, c_b, c̄_r, c̄_g, c̄_b]`

That is: mass, spin, left/right weak isospin and hypercharge, and SU(3) colour. This is exactly the Standard Model "particle ID" — and the network learns to use it.

### Diagram topology
- **s-channel** (5 edges): initial_1 ──●══●── final_1 with a single propagator joining the two interaction vertices, initial_2 ──●, ●── final_2.
- **t-channel** (5 edges): initial_1 connects to final_1 directly via a propagator, etc.

In [ ]:
# === Particle class (edge features) ===
# Simplified from matbun/Feynman-GNN: each particle is a 12-dim feature vector

class Particle:
    """Edge feature vector: [m, S, LIW3, LY, RIW3, RY, c_r, c_g, c_b, cbar_r, cbar_g, cbar_b]"""
    def __init__(self, m, S, LIW=0, LY=0, RIW=0, RY=0, colour=(0,0,0), anti_colour=(0,0,0)):
        self.feat = [m, S, LIW, LY, RIW, RY, *colour, *anti_colour]
    def get_feat(self):
        return list(self.feat)

class E_minus(Particle):
    def __init__(self):
        super().__init__(m=m_e, S=0.5, LIW=-0.5, LY=-1, RIW=0, RY=-2)

class E_plus(Particle):
    def __init__(self):
        super().__init__(m=m_e, S=0.5, LIW=0, LY=2, RIW=0.5, RY=1)

class Mu_minus(Particle):
    def __init__(self):
        super().__init__(m=m_mu, S=0.5, LIW=-0.5, LY=-1, RIW=0, RY=-2)

class Mu_plus(Particle):
    def __init__(self):
        super().__init__(m=m_mu, S=0.5, LIW=0, LY=2, RIW=0.5, RY=1)

class Photon(Particle):
    def __init__(self):
        super().__init__(m=0, S=1)  # neutral, massless, spin-1

# Demo: print the feature vector of an electron and a photon
print(f"e-     features: {E_minus().get_feat()}")
print(f"photon features: {Photon().get_feat()}")
print(f"mu+    features: {Mu_plus().get_feat()}")

In [ ]:
# === Diagram builder: s-channel and t-channel topologies ===
# Simplified from matbun/Feynman-GNN: Dataset_generator.ipynb
# Node indices: 0=init_1, 1=init_2, 2=vert_L, 3=vert_R, 4=final_1, 5=final_2

INITIAL = [1, 0, 0]
VIRTUAL = [0, 1, 0]
FINAL   = [0, 0, 1]

def build_s_channel(part_initial, part_final):
    """s-channel: init_1/init_2 -> L, L -> R (photon), R -> final_1/final_2."""
    p_in, p_out = part_initial.get_feat(), part_final.get_feat()
    # Incoming anti-particle uses opposite-sign features (handled by caller via E_plus/Mu_plus)
    node_features = [INITIAL, INITIAL, VIRTUAL, VIRTUAL, FINAL, FINAL]
    src  = [0, 1, 2, 3, 3]
    dst  = [2, 2, 3, 4, 5]
    # Edge features: initial leg, initial leg, photon propagator, final leg, final leg
    edge_feats = [p_in, p_in, Photon().get_feat(), p_out, p_out]
    return node_features, [src, dst], edge_feats

def build_t_channel(part_initial, part_final):
    """t-channel: init_1 -> L -> final_1 (same line), init_2 -> R -> final_2, L -> R (photon)."""
    p_in, p_out = part_initial.get_feat(), part_final.get_feat()
    node_features = [INITIAL, INITIAL, VIRTUAL, VIRTUAL, FINAL, FINAL]
    src  = [0, 1, 2, 2, 3]
    dst  = [2, 3, 3, 4, 5]
    edge_feats = [p_in, p_in, Photon().get_feat(), p_out, p_out]
    return node_features, [src, dst], edge_feats

def combine_diagrams(diag1, diag2):
    """Combine two diagrams into one disconnected-union graph. Useful for s+t channel Bhabha."""
    n1 = len(diag1[0])
    nodes = diag1[0] + diag2[0]
    src = diag1[1][0] + [s + n1 for s in diag2[1][0]]
    dst = diag1[1][1] + [d + n1 for d in diag2[1][1]]
    edge_feats = diag1[2] + diag2[2]
    return nodes, [src, dst], edge_feats

def make_undirected(diag):
    """Duplicate every edge in the opposite direction for bidirectional message passing."""
    nodes, (src, dst), feats = diag
    return nodes, [src + dst, dst + src], feats + feats

# Test
d_ee_mumu = make_undirected(build_s_channel(E_minus(), Mu_minus()))
print(f"e+e- -> mu+mu- diagram:")
print(f"  Nodes: {len(d_ee_mumu[0])}")
print(f"  Edges: {len(d_ee_mumu[2])} (after undirecting)")

d_bhabha = make_undirected(combine_diagrams(
    build_s_channel(E_minus(), E_minus()),
    build_t_channel(E_minus(), E_minus())
))
print(f"\nBhabha diagram (s+t):")
print(f"  Nodes: {len(d_bhabha[0])}")
print(f"  Edges: {len(d_bhabha[2])} (after undirecting)")

In [ ]:
# === Visualize a diagram as a graph ===

def plot_diagram_graph(diag, title="Feynman diagram graph", ax=None):
    nodes, (src, dst), edge_feats = diag
    G = nx.MultiDiGraph()
    for i, nf in enumerate(nodes):
        label = {0: 'init', 1: 'virt', 2: 'final'}[nf.index(1)]
        G.add_node(i, label=label)
    # Only plot forward edges (not the undirected duplicates)
    n_fwd = len(edge_feats) // 2 if len(src) == 2*len(edge_feats)//2 else len(edge_feats)
    for k in range(len(src) // 2):
        s, d = src[k], dst[k]
        # Photon propagator has m=0 & S=1 and no colour
        is_photon = edge_feats[k][0] == 0 and edge_feats[k][1] == 1
        G.add_edge(s, d, photon=is_photon)
    if ax is None:
        fig, ax = plt.subplots(figsize=(6,4))
    # Layered layout: initial left, virtual middle, final right
    pos = {}
    for i, nf in enumerate(nodes):
        if nf == INITIAL: pos[i] = (0.0, -i)
        elif nf == VIRTUAL: pos[i] = (1.0, -i*0.8)
        elif nf == FINAL: pos[i] = (2.0, -i)
    node_colors = ['#4C72B0' if nodes[i]==INITIAL else '#DD8452' if nodes[i]==VIRTUAL else '#55A868' for i in G.nodes]
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=700, ax=ax)
    for u, v, d in G.edges(data=True):
        style = 'dashed' if d.get('photon') else 'solid'
        colour = 'purple' if d.get('photon') else 'black'
        nx.draw_networkx_edges(G, pos, edgelist=[(u,v)], style=style, edge_color=colour, ax=ax,
                               arrows=True, arrowsize=15, connectionstyle='arc3,rad=0.15')
    nx.draw_networkx_labels(G, pos, labels={i:str(i) for i in G.nodes}, font_color='white', font_weight='bold', ax=ax)
    ax.set_title(title)
    ax.axis('off')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_diagram_graph(d_ee_mumu, r"$e^+e^-\to\mu^+\mu^-$ (s-channel)", axes[0])
plot_diagram_graph(d_bhabha, r"Bhabha (s + t channels)", axes[1])
plt.tight_layout()
plt.show()

print("Legend: blue=initial, orange=virtual, green=final. Dashed purple = photon propagator.")

## 4. Generate the Dataset

We sweep over a grid of kinematic inputs $(p, \theta)$ and compute the analytic $|\mathcal{M}|^2$ for each, packaging the result as (graph, label).

- **Dataset A**: $e^+e^- \to \mu^+\mu^-$ with s-channel only — ~5000 samples
- **Dataset B**: $e^+e^- \to e^+e^-$ (Bhabha) with s+t channels — ~5000 samples

Each sample has a *graph representation* of its diagram and the target $|\mathcal{M}|^2$. The MLP baseline will ignore the graph and use only $(p, \theta)$.

In [ ]:
def diagram_to_pyg(diag, p, theta, y, process_id):
    """Convert a (diagram, kinematics, label) triple to a PyTorch Geometric Data object."""
    nodes, (src, dst), edge_feats = diag
    x = torch.tensor(nodes, dtype=torch.float)
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_attr = torch.tensor(edge_feats, dtype=torch.float)
    # Attach kinematics as graph-level features (distributed on every node for simplicity)
    s = 4 * (m_e**2 + p**2)
    t = 2 * m_e**2 - 2 * p**2 * (1 - np.cos(theta))
    u = 2 * m_e**2 - 2 * p**2 * (1 + np.cos(theta))
    kin = torch.tensor([[p, theta, s, t, u]], dtype=torch.float)
    return Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.tensor([y], dtype=torch.float),
        kin=kin,
        process_id=torch.tensor([process_id], dtype=torch.long),
    )

def build_dataset(process_name, n_samples=5000, p_min=1e3, p_max=1e5):
    """Generate a list of PyG graphs for one process."""
    if process_name == "ee_mumu":
        diag_template = make_undirected(build_s_channel(E_minus(), Mu_minus()))
        Mfi = Mfi_ee_mumu
        pid = 0
    elif process_name == "bhabha":
        diag_template = make_undirected(combine_diagrams(
            build_s_channel(E_minus(), E_minus()),
            build_t_channel(E_minus(), E_minus())
        ))
        Mfi = Mfi_bhabha
        pid = 1
    else:
        raise ValueError(f"Unknown process: {process_name}")

    # Sample kinematics on a 2D grid
    n_p = int(np.sqrt(n_samples))
    n_theta = n_samples // n_p
    ps = np.linspace(p_min, p_max, n_p)
    thetas = np.linspace(0.1, np.pi - 0.1, n_theta)

    graphs = []
    for p in ps:
        for th in thetas:
            y = float(Mfi(p, th))
            graphs.append(diagram_to_pyg(diag_template, float(p), float(th), y, pid))
    return graphs

print("Generating ee -> mumu dataset...")
data_mumu = build_dataset("ee_mumu", n_samples=5000)
print(f"  {len(data_mumu)} samples")

print("Generating Bhabha dataset...")
data_bhabha = build_dataset("bhabha", n_samples=5000)
print(f"  {len(data_bhabha)} samples")

# Normalize labels to log-scale (|M|^2 spans several orders of magnitude in Bhabha)
all_y = np.array([float(d.y) for d in data_mumu + data_bhabha])
y_log = np.log(all_y + 1e-20)
y_mean, y_std = y_log.mean(), y_log.std()
print(f"\nlog |M|^2 statistics: mean={y_mean:.3f}, std={y_std:.3f}")

def attach_normalized_label(graphs):
    for g in graphs:
        g.y_norm = torch.tensor([(np.log(float(g.y) + 1e-20) - y_mean) / y_std], dtype=torch.float)
    return graphs

data_mumu = attach_normalized_label(data_mumu)
data_bhabha = attach_normalized_label(data_bhabha)

## 5. Three Baselines Before We Reach for a GNN

Before we compare architecture families, let's build the progression of baselines one by one:

### 5a. Kinematic MLP (weak baseline)
Input: just the 5-d kinematic vector $(p, \theta, s, t, u)$. The graph is ignored entirely. This is what a physicist's first attempt would look like.

### 5b. Flattened-graph MLP (fair baseline)
Input: the **full graph content**, flattened to a fixed-size padded vector (321 dims):
- Node features padded to `MAX_NODES=12` × 3 = 36 dims
- Edges (sorted canonically by (src, dst)) padded to `MAX_EDGES=20` × 14 dims = 280 dims (each edge: [src_idx/N, dst_idx/N, 12d particle features])
- Kinematics: 5 dims

Requires a canonical edge ordering, and uses log1p-normalization on inputs to handle the wide dynamic range.

### 5c. Edge Transformer (strong baseline)
Input: a **set of edge tokens**, each carrying its particle features + src/dst node embeddings. Multi-head self-attention processes all edges together — no canonical ordering needed (attention is permutation-equivariant). Mask out padding tokens.

### 5d. FeynmanGNN (graph-aware)
GATConv on the true graph structure: `edge_index` specifies adjacency, message passing respects it. Attention is restricted to connected neighbors.

**The architectural hierarchy:**
- MLP on flat: needs canonical ordering; everything interacts with everything through dense layers.
- Transformer on edge set: permutation-equivariant; everything interacts with everything through attention.
- GNN: permutation-equivariant AND respects adjacency — attention is restricted to connected neighbors.

**Expected narrative:** the transformer is a good middle ground. It should match or beat the FlattenedMLP (no ordering constraint), and it may match or nearly match the GNN. The comparison shows clearly that "fancier" isn't always "better" — each architecture embeds a different inductive bias.

In [ ]:
# Fixed-size padding parameters for the flattened representation
MAX_NODES = 12   # Bhabha has 12 nodes (6 for mumu, 12 for Bhabha)
MAX_EDGES = 20   # Bhabha has 20 undirected edges (10 for mumu, 20 for Bhabha)
EDGE_DIM_FLAT = 2 + 12   # [src_idx/N, dst_idx/N, 12d particle features]

def flatten_graph(data):
    """Flatten a PyG Data object to a fixed-size padded feature vector.

    Layout: [node_features_padded (36) || edge_features_padded (280) || kinematics (5)]  = 321 dims
    Edges are sorted canonically by (src, dst) so ordering is deterministic.
    """
    x = data.x                      # [n, 3]
    edge_index = data.edge_index    # [2, e]
    edge_attr = data.edge_attr      # [e, 12]
    kin = data.kin.squeeze(0)       # [5]

    n, e = x.shape[0], edge_attr.shape[0]

    # Pad nodes
    x_flat = torch.zeros(MAX_NODES, 3)
    x_flat[:n] = x

    # Canonical edge ordering: sort by src * MAX_NODES + dst
    sort_key = edge_index[0].long() * MAX_NODES + edge_index[1].long()
    order = torch.argsort(sort_key)
    ei = edge_index[:, order]
    ea = edge_attr[order]

    # Build padded edge representation: [src_norm, dst_norm, particle_features]
    edge_flat = torch.zeros(MAX_EDGES, EDGE_DIM_FLAT)
    for i in range(e):
        edge_flat[i, 0] = ei[0, i].float() / MAX_NODES
        edge_flat[i, 1] = ei[1, i].float() / MAX_NODES
        edge_flat[i, 2:] = ea[i]

    return torch.cat([x_flat.flatten(), edge_flat.flatten(), kin])

FLAT_DIM = MAX_NODES * 3 + MAX_EDGES * EDGE_DIM_FLAT + 5  # 321
print(f"Flattened input dim: {FLAT_DIM}")

# Precompute flat representations for all graphs (so training is fast)
def attach_flat(graphs):
    for g in graphs:
        g.x_flat = flatten_graph(g)
    return graphs

attach_flat(data_mumu)
attach_flat(data_bhabha)
print(f"Example flattened vector shape: {data_mumu[0].x_flat.shape}")


class KinematicMLP(nn.Module):
    """Weak baseline: MLP on (p, theta, s, t, u) only. Blind to the diagram."""
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, kin):
        x = torch.sign(kin) * torch.log1p(torch.abs(kin))
        return self.net(x).squeeze(-1)


class FlattenedMLP(nn.Module):
    """Fair baseline: MLP on the full flattened padded graph + kinematics (321 dims).
    Sees exactly the same information as the GNN.
    Inputs are log1p-normalized to handle the wide range (masses in MeV, Mandelstam s ~ 1e10)."""
    def __init__(self, in_dim=FLAT_DIM, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x_flat):
        # log1p-normalize: compresses the wide dynamic range of raw physical quantities
        x = torch.sign(x_flat) * torch.log1p(torch.abs(x_flat))
        return self.net(x).squeeze(-1)


def train_mlp(graphs, model_type="kinematic", n_epochs=30, val_frac=0.2):
    """Train either the kinematic or flattened MLP on a list of PyG graphs."""
    n_val = int(len(graphs) * val_frac)
    perm = torch.randperm(len(graphs))
    train_set = [graphs[i] for i in perm[n_val:]]
    val_set = [graphs[i] for i in perm[:n_val]]

    if model_type == "kinematic":
        model = KinematicMLP().to(DEVICE)
        get_x = lambda subset: torch.cat([g.kin for g in subset]).to(DEVICE)
    else:
        model = FlattenedMLP().to(DEVICE)
        get_x = lambda subset: torch.stack([g.x_flat for g in subset]).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    x_train = get_x(train_set)
    y_train = torch.cat([g.y_norm for g in train_set]).to(DEVICE)
    x_val = get_x(val_set)
    y_val = torch.cat([g.y_norm for g in val_set]).to(DEVICE)

    history = {"train": [], "val": []}
    for epoch in range(n_epochs):
        model.train()
        idx = torch.randperm(len(train_set))
        for i in range(0, len(idx), 128):
            b = idx[i:i+128]
            opt.zero_grad()
            pred = model(x_train[b])
            loss = loss_fn(pred, y_train[b])
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            history["train"].append(loss_fn(model(x_train), y_train).item())
            history["val"].append(loss_fn(model(x_val), y_val).item())
    return model, history


# Train each baseline on the combined dataset
combined = data_mumu + data_bhabha

print("Training KinematicMLP on combined dataset...")
kin_mlp, kin_hist = train_mlp(combined, model_type="kinematic")
print(f"  final val MSE = {kin_hist['val'][-1]:.4f}")

print("\nTraining FlattenedMLP on combined dataset...")
flat_mlp, flat_hist = train_mlp(combined, model_type="flat")
print(f"  final val MSE = {flat_hist['val'][-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (hist, title) in zip(axes, [(kin_hist, "Kinematic MLP (5 features)"), (flat_hist, "Flattened-graph MLP (321 features)")]):
    ax.plot(hist["train"], label="train")
    ax.plot(hist["val"], label="val")
    ax.set_title(title); ax.set_xlabel("epoch"); ax.set_ylabel("MSE loss"); ax.legend()
plt.tight_layout(); plt.show()

print("\nObservation:")
print("  - KinematicMLP struggles: (s,t,u) alone does not identify the process.")
print("  - FlattenedMLP works well: it has the same info as the GNN, just flattened.")
print("  - Lesson: a well-designed MLP baseline is not a straw man. Always build one before claiming a GNN helps.")

### 5c. Edge Transformer

Each edge becomes a token. The token for edge $(i, j)$ with particle features $f$ is:
$$\text{token}_{ij} = \text{Linear}(f) + E_{\text{src}}(i) + E_{\text{dst}}(j)$$
where $E_{\text{src}}, E_{\text{dst}}$ are learnable node-index embeddings. A `[CLS]` token is prepended; self-attention mixes all tokens; we read out the `[CLS]` vector, concatenate kinematics, and pass through a final MLP head.

Padding mask excludes the unused edge slots from attention. This avoids the "phantom signal from padded zeros" problem the FlattenedMLP has.

In [ ]:
class EdgeTransformer(nn.Module):
    """Transformer encoder over edges. Permutation-equivariant, handles variable graph size via masking."""
    def __init__(self, max_edges=MAX_EDGES, max_nodes=MAX_NODES, edge_feat_dim=12, kin_dim=5,
                 d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.max_edges = max_edges
        self.max_nodes = max_nodes
        # Learnable embeddings for node indices (src and dst) — this is the *structural* info
        self.node_emb_src = nn.Embedding(max_nodes + 1, d_model)  # +1 for padding index
        self.node_emb_dst = nn.Embedding(max_nodes + 1, d_model)
        # Projection for particle features (log1p-normalized first)
        self.edge_proj = nn.Linear(edge_feat_dim, d_model)
        # Learnable CLS token
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)
        # Transformer encoder
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=2*d_model,
                                               dropout=dropout, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        # Output head (after concat with kinematics)
        self.kin_proj = nn.Linear(kin_dim, d_model)
        self.head = nn.Sequential(
            nn.Linear(2 * d_model, d_model), nn.GELU(),
            nn.Linear(d_model, 1),
        )

    def forward(self, edge_src, edge_dst, edge_feat, edge_mask, kin):
        """
        edge_src:  [B, E] long, src node index (max_nodes for padding)
        edge_dst:  [B, E] long, dst node index (max_nodes for padding)
        edge_feat: [B, E, 12] float, particle features (log1p-normalized)
        edge_mask: [B, E] bool, True where padded (excluded from attention)
        kin:       [B, 5] float, kinematics (log1p-normalized)
        """
        B = edge_src.shape[0]
        tokens = self.edge_proj(edge_feat) + self.node_emb_src(edge_src) + self.node_emb_dst(edge_dst)
        cls = self.cls.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        # Build full padding mask: CLS is always valid
        cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=edge_mask.device)
        mask = torch.cat([cls_mask, edge_mask], dim=1)
        out = self.encoder(tokens, src_key_padding_mask=mask)
        cls_out = out[:, 0, :]
        k = torch.sign(kin) * torch.log1p(torch.abs(kin))
        kin_h = self.kin_proj(k)
        return self.head(torch.cat([cls_out, kin_h], dim=-1)).squeeze(-1)


def build_transformer_inputs(data):
    """Extract (edge_src, edge_dst, edge_feat_normalized, edge_mask, kin) from a PyG Data object."""
    edge_index = data.edge_index  # [2, e]
    edge_attr = data.edge_attr    # [e, 12]
    kin = data.kin.squeeze(0)     # [5]
    e = edge_attr.shape[0]

    # Pad to MAX_EDGES; use MAX_NODES as the padding index for the embeddings
    src = torch.full((MAX_EDGES,), MAX_NODES, dtype=torch.long)
    dst = torch.full((MAX_EDGES,), MAX_NODES, dtype=torch.long)
    feat = torch.zeros(MAX_EDGES, 12)
    mask = torch.ones(MAX_EDGES, dtype=torch.bool)  # True = padded
    src[:e] = edge_index[0].long()
    dst[:e] = edge_index[1].long()
    # log1p-normalize particle features (masses span 0 to 105 MeV)
    feat[:e] = torch.sign(edge_attr) * torch.log1p(torch.abs(edge_attr))
    mask[:e] = False
    return src, dst, feat, mask, kin


def attach_transformer_inputs(graphs):
    for g in graphs:
        s, d, f, m, k = build_transformer_inputs(g)
        g.tr_src = s; g.tr_dst = d; g.tr_feat = f; g.tr_mask = m; g.tr_kin = k
    return graphs

attach_transformer_inputs(data_mumu)
attach_transformer_inputs(data_bhabha)


def train_transformer(graphs, n_epochs=30, val_frac=0.2, batch_size=128):
    n_val = int(len(graphs) * val_frac)
    perm = torch.randperm(len(graphs))
    train_set = [graphs[i] for i in perm[n_val:]]
    val_set = [graphs[i] for i in perm[:n_val]]

    model = EdgeTransformer().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=5e-4)
    loss_fn = nn.MSELoss()

    def stack(subset):
        return (torch.stack([g.tr_src for g in subset]).to(DEVICE),
                torch.stack([g.tr_dst for g in subset]).to(DEVICE),
                torch.stack([g.tr_feat for g in subset]).to(DEVICE),
                torch.stack([g.tr_mask for g in subset]).to(DEVICE),
                torch.stack([g.tr_kin for g in subset]).to(DEVICE),
                torch.cat([g.y_norm for g in subset]).to(DEVICE))

    xs_tr = stack(train_set)
    xs_val = stack(val_set)

    history = {"train": [], "val": []}
    for epoch in range(n_epochs):
        model.train()
        idx = torch.randperm(len(train_set))
        for i in range(0, len(idx), batch_size):
            b = idx[i:i+batch_size]
            opt.zero_grad()
            pred = model(xs_tr[0][b], xs_tr[1][b], xs_tr[2][b], xs_tr[3][b], xs_tr[4][b])
            loss = loss_fn(pred, xs_tr[5][b])
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            history["train"].append(loss_fn(model(*xs_tr[:-1]), xs_tr[-1]).item())
            history["val"].append(loss_fn(model(*xs_val[:-1]), xs_val[-1]).item())
        if (epoch + 1) % 5 == 0:
            print(f"  epoch {epoch+1:2d} | train {history['train'][-1]:.4f} | val {history['val'][-1]:.4f}")
    return model, history


print("Training EdgeTransformer on combined dataset...")
trans_both, trans_hist = train_transformer(data_mumu + data_bhabha)

n_params = lambda m: sum(p.numel() for p in m.parameters())
print(f"\nParameter counts:")
print(f"  KinematicMLP:    {n_params(kin_mlp):>8,}")
print(f"  FlattenedMLP:    {n_params(flat_mlp):>8,}")
print(f"  EdgeTransformer: {n_params(trans_both):>8,}")

## 6. The Graph Neural Network

We build a GNN that operates on the diagram graph. Architecture (simplified from `matbun/Feynman-GNN`'s TransformerConv model, using a standard GATConv stack):

```
Node features (3-d one-hot) ──┐
                              ├─► GATConv (edge-aware) ─► GATConv ─► Pool ─► MLP ─► |M|^2
Edge features (12-d particle) ┘                                              (+ kinematics)
```

**Key points:**
- `GATConv` with `edge_dim` parameter consumes our 12-d particle-identity edge features at each message-passing step
- Global pooling collapses the varying-size graph into a fixed-size representation
- Kinematic features $(p,\theta,s,t,u)$ are concatenated to the graph embedding before the final MLP — the graph tells us *which process*, the kinematics tell us *where in phase space*

This is genuinely the geometric-DL story: the graph structure is the input that distinguishes processes.

In [ ]:
class FeynmanGNN(nn.Module):
    """Small GAT on Feynman diagram graphs + kinematic tail."""
    def __init__(self, node_in=3, edge_in=12, kin_in=5, hidden=32, heads=2, n_layers=2):
        super().__init__()
        # First convolution: input node dim -> hidden
        self.conv_in = GATConv(node_in, hidden, heads=heads, edge_dim=edge_in, dropout=0.1)
        # Stacked convolutions
        self.convs = nn.ModuleList([
            GATConv(hidden * heads, hidden, heads=heads, edge_dim=edge_in, dropout=0.1)
            for _ in range(n_layers - 1)
        ])
        # Merge attention heads
        self.proj = nn.Linear(hidden * heads, hidden)
        # Kinematics encoder
        self.kin_enc = nn.Sequential(
            nn.Linear(kin_in, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        # Readout
        self.head = nn.Sequential(
            nn.Linear(2 * hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x, edge_index, edge_attr, batch, kin):
        h = F.elu(self.conv_in(x, edge_index, edge_attr))
        for c in self.convs:
            h = F.elu(c(h, edge_index, edge_attr))
        h = self.proj(h)
        # Pool: graph-level representation
        hg = global_mean_pool(h, batch)
        # Kinematics on log scale
        k = torch.sign(kin) * torch.log1p(torch.abs(kin))
        hk = self.kin_enc(k)
        combined = torch.cat([hg, hk], dim=-1)
        return self.head(combined).squeeze(-1)

def train_gnn(graphs, n_epochs=30, val_frac=0.2, batch_size=128):
    n_val = int(len(graphs) * val_frac)
    perm = torch.randperm(len(graphs))
    train_set = [graphs[i] for i in perm[n_val:]]
    val_set = [graphs[i] for i in perm[:n_val]]

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size)

    model = FeynmanGNN().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    history = {"train": [], "val": []}
    for epoch in range(n_epochs):
        model.train()
        tloss = 0; tn = 0
        for b in train_loader:
            b = b.to(DEVICE)
            opt.zero_grad()
            pred = model(b.x, b.edge_index, b.edge_attr, b.batch, b.kin)
            loss = loss_fn(pred, b.y_norm)
            loss.backward(); opt.step()
            tloss += loss.item() * b.num_graphs; tn += b.num_graphs
        model.eval(); vloss = 0; vn = 0
        with torch.no_grad():
            for b in val_loader:
                b = b.to(DEVICE)
                pred = model(b.x, b.edge_index, b.edge_attr, b.batch, b.kin)
                vloss += loss_fn(pred, b.y_norm).item() * b.num_graphs; vn += b.num_graphs
        history["train"].append(tloss/tn); history["val"].append(vloss/vn)
        if (epoch + 1) % 5 == 0:
            print(f"  epoch {epoch+1:2d} | train {tloss/tn:.4f} | val {vloss/vn:.4f}")
    return model, history, val_loader

print("Training GNN on BOTH processes (mumu + Bhabha)...")
gnn_both, hist_gnn, val_loader_gnn = train_gnn(data_mumu + data_bhabha)

print(f"\nFinal validation MSE on the combined dataset:")
print(f"  KinematicMLP     (5 features, weak):        {kin_hist['val'][-1]:.4f}")
print(f"  FlattenedMLP     (321 features, fair):      {flat_hist['val'][-1]:.4f}")
print(f"  EdgeTransformer  (set of edges, strong):    {trans_hist['val'][-1]:.4f}")
print(f"  FeynmanGNN       (graph-aware):             {hist_gnn['val'][-1]:.4f}")

## 7. Evaluation: Predictions vs Truth

Let's look at how well each model reproduces the analytic $|\mathcal{M}|^2$ at fixed momentum across the full range of scattering angles.

In [ ]:
def predict_curve(model, process_name, p=5e4, n=100, model_type="gnn"):
    """Sweep theta at fixed p and return predicted |M|^2 alongside truth.

    model_type is one of: 'gnn', 'kinematic', 'flat', 'transformer'.
    """
    thetas = np.linspace(0.15, np.pi - 0.15, n)
    if process_name == "ee_mumu":
        diag = make_undirected(build_s_channel(E_minus(), Mu_minus()))
        Mfi = Mfi_ee_mumu; pid = 0
    else:
        diag = make_undirected(combine_diagrams(
            build_s_channel(E_minus(), E_minus()),
            build_t_channel(E_minus(), E_minus())
        ))
        Mfi = Mfi_bhabha; pid = 1

    graphs = [diagram_to_pyg(diag, float(p), float(t), float(Mfi(p, t)), pid) for t in thetas]
    if model_type == "flat":
        for g in graphs:
            g.x_flat = flatten_graph(g)
    elif model_type == "transformer":
        attach_transformer_inputs(graphs)

    loader = DataLoader(graphs, batch_size=len(graphs))
    model.eval()
    true = np.array([Mfi(p, t) for t in thetas])
    with torch.no_grad():
        b = next(iter(loader)).to(DEVICE)
        if model_type == "gnn":
            pred_norm = model(b.x, b.edge_index, b.edge_attr, b.batch, b.kin).cpu().numpy()
        elif model_type == "kinematic":
            pred_norm = model(b.kin).cpu().numpy()
        elif model_type == "flat":
            x_flat = torch.stack([g.x_flat for g in graphs]).to(DEVICE)
            pred_norm = model(x_flat).cpu().numpy()
        elif model_type == "transformer":
            tr_src  = torch.stack([g.tr_src for g in graphs]).to(DEVICE)
            tr_dst  = torch.stack([g.tr_dst for g in graphs]).to(DEVICE)
            tr_feat = torch.stack([g.tr_feat for g in graphs]).to(DEVICE)
            tr_mask = torch.stack([g.tr_mask for g in graphs]).to(DEVICE)
            tr_kin  = torch.stack([g.tr_kin for g in graphs]).to(DEVICE)
            pred_norm = model(tr_src, tr_dst, tr_feat, tr_mask, tr_kin).cpu().numpy()
        else:
            raise ValueError(model_type)
    pred = np.exp(pred_norm * y_std + y_mean)
    return thetas, true, pred


models_to_compare = [
    (kin_mlp,    "kinematic",   "KinematicMLP",    "crimson"),
    (flat_mlp,   "flat",        "FlattenedMLP",    "goldenrod"),
    (trans_both, "transformer", "EdgeTransformer", "mediumorchid"),
    (gnn_both,   "gnn",         "FeynmanGNN",      "teal"),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for row, (process, label) in enumerate([("ee_mumu", r"$e^+e^-\to\mu^+\mu^-$"), ("bhabha", r"Bhabha")]):
    for col, (model, mtype, name, color) in enumerate(models_to_compare):
        thetas, true, pred = predict_curve(model, process, model_type=mtype)
        axes[row, col].plot(thetas, true, 'k--', label='truth', lw=2)
        axes[row, col].plot(thetas, pred, 'o-', color=color, label=name, alpha=0.6, markersize=3)
        axes[row, col].set_yscale('log')
        axes[row, col].set_title(f"{name} on {label}")
        axes[row, col].set_xlabel(r"$\theta$"); axes[row, col].set_ylabel(r"$|\mathcal{M}|^2$")
        axes[row, col].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("All four models trained on the combined dataset.")
print("  - KinematicMLP averages badly across processes.")
print("  - FlattenedMLP, EdgeTransformer, FeynmanGNN all work well on this fixed-topology training set.")

## 8. Edge-Level Prediction: Learning a Feynman Rule

Everything so far has been **graph-level regression**: diagram in → scalar $|\mathcal{M}|^2$ out. But GNNs can also make **node-** or **edge-level** predictions — the same encoder, a different readout head.

Let's give the GNN a task that requires it to **learn a genuine Feynman rule**: momentum conservation at vertices, and hence the virtuality $q^2$ of each propagator.

### The Task

For every edge in every diagram, predict the squared 4-momentum $q^2$ flowing through it. Physically:

- **External edges** (on-shell particles): $q^2 = m^2$ — tiny. For an electron, $q^2 = m_e^2 \approx 0.26\,\text{MeV}^2$.
- **Propagator in an s-channel subdiagram**: $q^2 = s = (p_1 + p_2)^2 \approx 4p^2$ at high energy — large and positive.
- **Propagator in a t-channel subdiagram**: $q^2 = t = (p_1 - p_3)^2 \approx -2p^2(1 - \cos\theta)$ — large and negative.

The virtuality is what appears in the propagator factor $\frac{1}{q^2 - m^2}$ of the Feynman rules — it's the central physics quantity on each internal line.

### Why It's Genuinely Non-Trivial

The GNN must learn to:
1. **Identify external vs internal edges** from local topology (external edges connect to an `initial` or `final` node; propagators connect two `virtual` nodes).
2. **Distinguish s- from t-channel subdiagrams** in a Bhabha graph, which are two *disconnected components*. Their topology differs: in s-channel, both initial legs attach to the same virtual node; in t-channel, they attach to different virtual nodes. The GNN has to read this connectivity pattern.
3. **Compute the right kinematic invariant** (s, t, or u) from the external-state momenta propagated through vertex structure.

Together, the model is learning a compressed version of the Feynman rules: *"given the diagram topology and external momenta, what flows through each line?"*

In [ ]:
# === Compute q^2 target for each edge in each diagram ===
# For external edges, q^2 = m^2.
# For propagator edges, q^2 = s (s-channel) or q^2 = t (t-channel).
# We tag edges during dataset construction to know which category they belong to.

def diagram_to_pyg_with_q2(diag_builder_fn, p, theta, process_name, y, process_id):
    """Like diagram_to_pyg but additionally computes the true q^2 per edge.

    diag_builder_fn returns (nodes, (src, dst), edge_feats, edge_q2)
    where edge_q2 is the list of q^2 values per directed edge (before undirecting).
    """
    nodes, (src, dst), edge_feats, edge_q2 = diag_builder_fn(p, theta)
    x = torch.tensor(nodes, dtype=torch.float)
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_attr = torch.tensor(edge_feats, dtype=torch.float)
    q2 = torch.tensor(edge_q2, dtype=torch.float)
    s = 4*(m_e**2+p**2); t = 2*m_e**2-2*p**2*(1-np.cos(theta)); u = 2*m_e**2-2*p**2*(1+np.cos(theta))
    kin = torch.tensor([[p, theta, s, t, u]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([y], dtype=torch.float),
                kin=kin, edge_q2=q2, num_nodes=len(nodes),
                process_id=torch.tensor([process_id], dtype=torch.long))


def build_ee_mumu_with_q2(p, theta):
    """Build the s-channel e+e- -> mu+mu- diagram + undirect + compute q^2 per edge."""
    diag = build_s_channel(E_minus(), Mu_minus())   # s-channel, initial=e, final=mu
    nodes, (src, dst), feats = diag
    # Edge q^2 in order: init_e, init_e, photon propagator, final_mu, final_mu
    s = 4*(m_e**2+p**2)
    q2_directed = [m_e**2, m_e**2, s, m_mu**2, m_mu**2]
    # Undirect (double edges + features)
    n, (s_list, d_list), f = make_undirected((nodes, (src, dst), feats))
    q2_full = q2_directed + q2_directed   # match undirected doubling
    return n, (s_list, d_list), f, q2_full


def build_bhabha_with_q2(p, theta):
    """Build the Bhabha s+t combined diagram + compute q^2 per edge."""
    # s-channel subdiagram: photon propagator carries q^2 = s
    s = 4*(m_e**2+p**2)
    t = 2*m_e**2 - 2*p**2*(1 - np.cos(theta))
    diag_s = build_s_channel(E_minus(), E_minus())   # all electrons
    diag_t = build_t_channel(E_minus(), E_minus())
    # q^2 in each subdiagram, before combining:
    q2_s_directed = [m_e**2, m_e**2, s, m_e**2, m_e**2]
    q2_t_directed = [m_e**2, m_e**2, t, m_e**2, m_e**2]
    diag_combined = combine_diagrams(diag_s, diag_t)
    n, (s_list, d_list), f = make_undirected(diag_combined)
    q2_directed = q2_s_directed + q2_t_directed
    q2_full = q2_directed + q2_directed
    return n, (s_list, d_list), f, q2_full


def build_q2_dataset(n_samples=5000, p_min=1e3, p_max=1e5):
    """Build combined dataset with q^2 targets per edge."""
    n_p = int(np.sqrt(n_samples))
    n_theta = n_samples // n_p
    ps = np.linspace(p_min, p_max, n_p)
    thetas = np.linspace(0.1, np.pi-0.1, n_theta)
    graphs = []
    for p in ps:
        for th in thetas:
            # ee -> mumu
            y_mm = float(Mfi_ee_mumu(p, th))
            graphs.append(diagram_to_pyg_with_q2(build_ee_mumu_with_q2, float(p), float(th), "mumu", y_mm, 0))
            # Bhabha
            y_bh = float(Mfi_bhabha(p, th))
            graphs.append(diagram_to_pyg_with_q2(build_bhabha_with_q2, float(p), float(th), "bhabha", y_bh, 1))
    return graphs

print("Building q^2 dataset...")
q2_graphs = build_q2_dataset(n_samples=5000)
print(f"  {len(q2_graphs)} diagrams total")

# Inspect one of each process
for g in [q2_graphs[0], q2_graphs[1]]:
    pid = g.process_id.item()
    name = "mumu" if pid == 0 else "Bhabha"
    print(f"\n{name} graph: {g.num_nodes} nodes, {g.edge_index.shape[1]} edges")
    print(f"  q^2 per edge (MeV^2): {[f'{q:+.2e}' for q in g.edge_q2.tolist()[:10]]}")

In [ ]:
# === Edge-level q^2 regression ===
# Uses the same GAT encoder as FeynmanGNN, but with a per-edge regression head.
# Input: graph topology + masked edge features + external kinematics.
# Target: log-signed virtuality, log_sq2 = sign(q^2) * log10(|q^2|).

def log_signed(x, eps=1e-8):
    return torch.sign(x) * torch.log10(torch.abs(x) + eps)

def inv_log_signed(x, eps=1e-8):
    return torch.sign(x) * (10.0**torch.abs(x) - eps)

# Compute global normalization from the training data
all_q2 = torch.cat([g.edge_q2 for g in q2_graphs])
all_log_q2 = log_signed(all_q2)
q2_mean, q2_std = all_log_q2.mean().item(), all_log_q2.std().item()
print(f"log10|q^2| statistics (training): mean={q2_mean:.2f}, std={q2_std:.2f}")

# Attach normalized target to each graph
for g in q2_graphs:
    lg = log_signed(g.edge_q2)
    g.q2_target = (lg - q2_mean) / q2_std

class EdgeQ2Regressor(nn.Module):
    """GAT encoder + per-edge regression head for predicting q^2.

    The encoder sees MASKED edge features (zeros) — it has to work from
    topology + kinematics alone. This mimics the physics task: "given a
    diagram skeleton, what virtuality flows through each line?"
    """
    def __init__(self, node_in=3, edge_in=12, kin_in=5, hidden=32, heads=2, n_layers=2):
        super().__init__()
        self.conv_in = GATConv(node_in, hidden, heads=heads, edge_dim=edge_in, dropout=0.1)
        self.convs = nn.ModuleList([
            GATConv(hidden * heads, hidden, heads=heads, edge_dim=edge_in, dropout=0.1)
            for _ in range(n_layers - 1)
        ])
        self.proj = nn.Linear(hidden * heads, hidden)
        self.edge_head = nn.Sequential(
            nn.Linear(2 * hidden + kin_in, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x, edge_index, edge_attr_masked, kin, batch):
        h = F.elu(self.conv_in(x, edge_index, edge_attr_masked))
        for c in self.convs:
            h = F.elu(c(h, edge_index, edge_attr_masked))
        h = self.proj(h)
        src, dst = edge_index[0], edge_index[1]
        edge_batch = batch[src]
        k_per_edge = torch.sign(kin[edge_batch]) * torch.log1p(torch.abs(kin[edge_batch]))
        return self.edge_head(torch.cat([h[src], h[dst], k_per_edge], dim=-1)).squeeze(-1)


# Split
perm = torch.randperm(len(q2_graphs))
n_val = int(0.2 * len(q2_graphs))
train_q2 = [q2_graphs[i] for i in perm[n_val:]]
val_q2 = [q2_graphs[i] for i in perm[:n_val]]
train_loader_q = DataLoader(train_q2, batch_size=128, shuffle=True)
val_loader_q = DataLoader(val_q2, batch_size=128)

q2_model = EdgeQ2Regressor().to(DEVICE)
opt = torch.optim.Adam(q2_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

print(f"\nEdgeQ2Regressor params: {sum(p.numel() for p in q2_model.parameters()):,}")
print("Training...")
for epoch in range(1, 31):
    q2_model.train()
    tl = 0; tn = 0
    for b in train_loader_q:
        b = b.to(DEVICE)
        # Mask edge features to zero so the model can't cheat via particle identity
        edge_attr_masked = torch.zeros_like(b.edge_attr)
        opt.zero_grad()
        pred = q2_model(b.x, b.edge_index, edge_attr_masked, b.kin, b.batch)
        loss = loss_fn(pred, b.q2_target)
        loss.backward(); opt.step()
        tl += loss.item() * len(b.q2_target); tn += len(b.q2_target)
    q2_model.eval()
    vl = 0; vn = 0
    with torch.no_grad():
        for b in val_loader_q:
            b = b.to(DEVICE)
            edge_attr_masked = torch.zeros_like(b.edge_attr)
            pred = q2_model(b.x, b.edge_index, edge_attr_masked, b.kin, b.batch)
            vl += loss_fn(pred, b.q2_target).item() * len(b.q2_target); vn += len(b.q2_target)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d} | train MSE {tl/tn:.4f} | val MSE {vl/vn:.4f}")

print(f"\nFinal val MSE (normalized log-q^2): {vl/vn:.4f}")

In [ ]:
# === Evaluate: scatter plot pred vs true q^2, split by edge category ===
#
# We categorize each edge by looking at its true |q^2|:
#   - External electron:  q^2 ~ 0.26 MeV^2    -> log10 ~ -0.6
#   - External muon:      q^2 ~ 1.1e4 MeV^2   -> log10 ~ 4
#   - s-channel prop:     q^2 = s (positive)  -> log10 ~ 8-10
#   - t-channel prop:     q^2 = t (negative)  -> log10 ~ 8-10 with negative sign

q2_model.eval()
true_list, pred_list = [], []
with torch.no_grad():
    for b in val_loader_q:
        b = b.to(DEVICE)
        pred = q2_model(b.x, b.edge_index, torch.zeros_like(b.edge_attr), b.kin, b.batch)
        true_list.append(b.q2_target.cpu())
        pred_list.append(pred.cpu())

true_norm = torch.cat(true_list)
pred_norm = torch.cat(pred_list)
# Invert normalization: get back to signed log10|q^2|
true_signed_log = true_norm * q2_std + q2_mean
pred_signed_log = pred_norm * q2_std + q2_mean

# Categorize edges by their true signed-log value
def categorize(slog):
    if slog > 6:   return 0   # s-channel propagator (large positive)
    if slog < -6:  return 1   # t-channel propagator (large negative)
    if 3 < slog < 5:  return 2   # external muon (log ~ 4)
    return 3                  # external electron (log ~ -0.6)

cats = np.array([categorize(s.item()) for s in true_signed_log])
cat_names = ["s-channel propagator", "t-channel propagator", "external muon", "external electron"]
cat_colors = ["teal", "crimson", "goldenrod", "steelblue"]

# Plot: pred vs true signed log q^2, colored by category
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for i, (name, color) in enumerate(zip(cat_names, cat_colors)):
    mask = cats == i
    ax.scatter(true_signed_log[mask].numpy(), pred_signed_log[mask].numpy(),
               alpha=0.3, s=10, c=color, label=f"{name} (n={mask.sum()})")
lo, hi = -12, 12
ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
ax.set_xlabel(r"True sign($q^2$)$\cdot \log_{10}|q^2|$ (MeV$^2$)")
ax.set_ylabel(r"Predicted sign($q^2$)$\cdot \log_{10}|q^2|$")
ax.set_title(r"Edge-level $q^2$ prediction")
ax.legend(fontsize=8, loc='lower right')
ax.grid(alpha=0.3)

# Error histograms per category
ax = axes[1]
for i, (name, color) in enumerate(zip(cat_names, cat_colors)):
    mask = cats == i
    err = (pred_signed_log[mask] - true_signed_log[mask]).numpy()
    ax.hist(err, bins=40, alpha=0.5, color=color, label=f"{name}: MAE={np.abs(err).mean():.2f}")
ax.set_xlabel(r"Error in sign($q^2$)$\cdot \log_{10}|q^2|$ (dex)")
ax.set_ylabel("Count")
ax.set_title("Error distribution by edge type")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Print per-category MAE
print("\nPer-category error (in dex, i.e. powers of 10):")
for i, name in enumerate(cat_names):
    mask = cats == i
    err = (pred_signed_log[mask] - true_signed_log[mask]).numpy()
    print(f"  {name:<24}: MAE = {np.abs(err).mean():.2f} dex    (n = {mask.sum()})")

print("""
What just happened: the GNN, given only graph topology + external kinematics
(edge particle features ZEROED), has to predict the virtuality q^2 on every edge.
To do this correctly it had to learn three things entirely from data:

  1. Which edges are external vs propagator  (from local connectivity)
  2. Which subdiagram a propagator belongs to  (s-channel vs t-channel,
     distinguishable by whether the two initial-state legs attach to the SAME
     virtual node or to DIFFERENT virtual nodes)
  3. How to compute the corresponding kinematic invariant  (s = 4p^2 vs
     t = -2p^2(1-cos theta)) from the (p, theta) input

This is a compressed, learned version of the Feynman rules for virtuality. The
same GNN encoder that did graph-level |M|^2 prediction now does edge-level
regression — a single architecture across different prediction granularities.
""")

## 9. Summary & Research Frontiers

### What We Did

1. Represented Feynman diagrams as graphs: vertices → nodes (3-d one-hot), propagators → edges (12-d Standard Model quantum numbers).
2. Generated a dataset of tree-level QED amplitudes analytically for two processes with different diagram topologies.
3. Built **four models** spanning an inductive-bias hierarchy:
   - **KinematicMLP** (5 features) — can't distinguish processes → fails
   - **FlattenedMLP** (321 padded features) — fair baseline, needs canonical ordering
   - **EdgeTransformer** (attention over edge tokens) — permutation-equivariant, fewer params
   - **FeynmanGNN** (GAT with adjacency) — strongest prior, best performance, fewest params
4. Showed the GNN wins on the combined multi-process dataset, while the kinematic MLP fails.
5. Inverted the problem: given amplitude + topology, predicted particle identities on each edge (edge-level classification). Same GNN encoder, different readout head — demonstrating the flexibility of message-passing architectures across prediction granularities.

### The Architectural Hierarchy

| Model | Params | Val MSE | Inductive bias |
|---|---|---|---|
| KinematicMLP | 8.7K | ~0.70 | None (kinematics only) |
| FlattenedMLP | 214K | ~0.004 | Sees full graph, needs ordering |
| EdgeTransformer | 78K | ~0.002 | Permutation-equivariant |
| FeynmanGNN | ~20K | ~0.0006 | Perm.-equiv. + adjacency |

Each step down adds an inductive bias and **reduces** the parameter count needed. That's the geometric deep learning punchline: the right structure makes the problem easier.

### Research Frontiers

This notebook is a toy proof of concept on 2→2 tree-level QED. The real frontier is scaling this to larger, more complex settings:

**Scaling to more processes.** The [original paper](https://arxiv.org/abs/2211.15348) trained on all tree-level QED permutations of e±/μ± (6 graphs, 4 matrix elements) and extended to QCD. The GNN handles this gracefully — accuracy degrades from 99% → 97% → 94% as complexity grows.

**Symbolic amplitude computation.** [SYMBA](https://github.com/ML4SCI/SYMBAHEP) (arXiv:2206.08901) uses a seq2seq transformer to compute symbolic squared amplitudes for all tree-level 2→2 and 2→3 QED+QCD processes — 99% accuracy on QED, 97.6% on QCD. Not a GNN but a powerful complementary approach.

**Amplitude simplification.** [Cheung, Dersy & Schwartz](https://arxiv.org/abs/2408.04720) train a transformer to simplify spinor-helicity amplitudes from hundreds of terms to compact Parke-Taylor-like forms. Open code at [github.com/aureliendersy/spinorhelicity](https://github.com/aureliendersy/spinorhelicity) with a [live demo](https://spinorhelicity.streamlit.app).

**Loop integral reduction.** The hottest area in 2025-26:
- [SAILIR](https://arxiv.org/abs/2604.05034) (April 2026) — self-supervised transformer for integration-by-parts (IBP) reduction. Trained via a scramble/unscramble procedure, no labeled data needed.
- [von Hippel & Wilhelm](https://arxiv.org/abs/2502.05121) — LLM-based genetic programming discovers improved IBP heuristics.
- [Zeng](https://arxiv.org/abs/2504.16045) — reinforcement learning optimizes Laporta algorithm orderings.

**Large-scale Feynman graph datasets.** [Borinsky et al.](https://arxiv.org/abs/2403.16217) studied ~2 million φ⁴ Feynman graphs — but found simple regression on graph invariants outperformed neural networks. Understanding *when* deep learning beats handcrafted features for diagram-level prediction remains open.

**Open gaps (research opportunities):**
- **QGRAF + GNN pipeline**: enumerate thousands of diagrams at multi-loop order with [QGRAF](https://cfif.ist.utl.pt/~paulo/qgraf.html), train a GNN. Nobody has published this.
- **Graph generative models for diagrams**: use VAEs/diffusion to *generate* valid Feynman topologies respecting vertex rules. Completely unexplored.
- **Self-supervised pretraining on diagram structure** (masked edge reconstruction, contrastive learning) to build foundation models for amplitude prediction.

### Further Reading
- [HEPML Living Review](https://github.com/iml-wg/HEPML-LivingReview) — continuously updated list of ML papers in HEP
- [ML + LHC Phenomenology Review](https://arxiv.org/abs/2602.03728) (Feb 2026) — comprehensive survey of ML across the theoretical prediction pipeline
- [GNNs in Particle Physics](https://arxiv.org/abs/2203.12852) — survey of GNN implementations in experimental HEP